# 生成式AI應用開發 第 07 週｜Streamlit Web App 入門

<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>🎯 本週目標</b>：把前幾週在 Colab 學到的 OpenAI 程式（問答、串流、對話記憶），<b>做成一個可以在瀏覽器操作、而且能部署上線的 Web App</b>。<br>
使用工具：<b>OpenAI Responses API + Streamlit</b>。
</div>

> 🧩 <b>本檔為 Claude Code 產出的教師版</b>，程式碼與練習均附完整參考答案。


## 📌 這週在整個課程的位置

| 週次 | 主題 | 你已經會的 |
|------|------|-----------|
| 第 3 週 | OpenAI API 入門 | `client.responses.create()`、`ask_ai_safe()` |
| 第 4 週 | Prompt Engineering | system prompt / role、few-shot |
| 第 5 週 | 對話狀態與 Streaming | 維護 `messages` history、逐字串流輸出 |
| 第 6 週 | Structured Outputs | JSON Schema、資料抽取 |
| **第 7 週** | **Streamlit Web App** | **← 把上面全部搬到網頁上** |

<div style="background-color:#fff8e6; border-left:6px solid #f0ad4e; padding:12px 16px; border-radius:6px;">
<b>💡 重點銜接</b>：第 5 週我們在 notebook 裡用 <code>for</code> 迴圈把串流一個字一個字印出來；這週會用 Streamlit 的 <code>st.write_stream()</code> 把<b>同一件事</b>顯示在網頁上。核心觀念一樣，只是換了呈現介面。
</div>


## 🔄 心態轉換：從 Colab 到 VS Code

<div style="background-color:#f0fff4; border-left:6px solid #5cb85c; padding:12px 16px; border-radius:6px;">
從這週開始，課程的開發方式會有一個重要轉變：
</div>

| | 前 6 週（Colab） | 第 7 週起（Streamlit） |
|---|---|---|
| 程式檔 | `.ipynb` notebook | `.py` 純 Python 檔 |
| 執行方式 | 一格一格按執行 | 在終端機輸入 `streamlit run app.py` |
| 成果 | 執行結果印在 cell 下方 | 一個網頁 App（有輸入框、按鈕） |
| 金鑰管理 | Colab Secrets | 本機 `.env` / 雲端 `st.secrets` |
| 部署 | 不需要 | 推上 GitHub → Streamlit Cloud 上線 |

> 🖥️ 建議工具：**VS Code**（本機開發）。若本機環境還沒裝好，本教材第 3 節提供 **Colab 臨時執行 Streamlit** 的備援方法。


## 1️⃣ Streamlit 是什麼？它怎麼跑？

**Streamlit** 是一個用純 Python 就能做出互動網頁的套件，不需要學 HTML / CSS / JavaScript。

```python
import streamlit as st
st.title("我的 App")
st.write("Hello, world!")
```

存成 `app.py`，在終端機執行：

```bash
streamlit run app.py
```

瀏覽器就會打開 `http://localhost:8501`。

<div style="background-color:#fdeeee; border-left:6px solid #d9534f; padding:12px 16px; border-radius:6px;">
<b>⚠️ 最重要的觀念：整個腳本會「從頭到尾重跑」</b><br>
每當使用者<b>按一個按鈕、輸入文字、切換選項</b>，Streamlit 都會<b>把整個 <code>app.py</code> 從第一行重新執行一次</b>。<br>
所以「要被記住的東西」（例如聊天紀錄）不能放普通變數，要放進 <code>st.session_state</code>（第 6 節會用到）。
</div>


## 2️⃣ 安裝環境

本機（VS Code 終端機）或 Colab 都先安裝三個套件：

- `streamlit`：做網頁
- `openai`：呼叫模型（延續前幾週）
- `python-dotenv`：讀取本機 `.env` 的 API key


In [ ]:
# 安裝本週需要的套件（Colab 或本機終端機皆可）
%pip install -q streamlit openai python-dotenv
print("套件安裝完成 ✅")


## 🪄 關於 `%%writefile`：用 notebook 產生 .py 檔

Streamlit App 必須是 `.py` 檔，不能直接在 notebook 裡「按執行」看到網頁。

為了讓你在這個 notebook 裡邊學邊產生檔案，我們用 Jupyter/Colab 的魔術指令 **`%%writefile 檔名.py`**：把某個 code cell 的內容<b>寫成一個 `.py` 檔</b>。

```python
%%writefile hello.py
print("這行會被寫進 hello.py，而不是馬上執行")
```

> 📝 執行完 `%%writefile` 之後，檔案就會出現在左側檔案總管。接著才用 `streamlit run` 去跑它。


## 3️⃣ 第一個最小 App

先做一個「輸入問題 → 按送出 → 顯示你剛剛打的字」的骨架，還沒接 AI。
重點是熟悉 4 個最常用元件：`st.title` / `st.text_input` / `st.button` / `st.write`。


In [ ]:
%%writefile app_1_hello.py
import streamlit as st

st.title("我的第一個 App")

user_input = st.text_input("請輸入問題：")

if st.button("送出") and user_input:
    st.write("你輸入的是：", user_input)


### ▶️ 怎麼執行（本機 / VS Code）

在專案資料夾的終端機輸入：

```bash
streamlit run app_1_hello.py
```

瀏覽器會自動開啟。改完程式存檔後，網頁右上角會出現 **Rerun**，按下即可更新。


### ☁️ 備援：在 Colab 臨時跑 Streamlit

如果本機環境還沒準備好，可以在 Colab 用 **localtunnel** 開一個臨時網址。
（正式開發仍建議用 VS Code；這只是應急。）


In [ ]:
# 只在 Colab 需要；本機請忽略這格。
# 執行後會出現一個 https://xxxx.loca.lt 的臨時網址，
# 點進去輸入下方印出的 Tunnel Password（就是這台機器的對外 IP）即可看到 App。
!npm install -q localtunnel
!streamlit run app_1_hello.py &>/content/logs.txt &
!npx localtunnel --port 8501 & curl -s https://loca.lt/mytunnelpassword


## 4️⃣ 接上 OpenAI：第一個 AI 問答 App

現在把第 3 週的 `ask_ai_safe()` 搬進來。這裡出現兩個新重點：

1. **`get_secret()`**：本機讀 `.env`、雲端讀 `st.secrets`，一段程式兩邊都能用。
2. **`with st.spinner(...)`**：等待 API 回覆時顯示「思考中…」。

> 延續主線：使用 OpenAI **Responses API**（`client.responses.create()` + `response.output_text`），預設模型 `gpt-5.4-mini`。


In [ ]:
%%writefile app_2_ai.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI


def get_secret(name, default=None):
    # 先讀雲端 Secrets，找不到再讀本機 .env
    try:
        if name in st.secrets:
            return st.secrets[name]
    except Exception:
        pass
    load_dotenv()
    return os.getenv(name, default)


def ask_ai_safe(user_input, role="你是有幫助的 AI 助理，請用繁體中文回答。"):
    api_key = get_secret("OPENAI_API_KEY")
    if not api_key:
        return "尚未設定 OPENAI_API_KEY。"
    client = OpenAI(api_key=api_key)
    model = get_secret("OPENAI_MODEL", "gpt-5.4-mini")
    try:
        response = client.responses.create(
            model=model,
            instructions=role,
            input=user_input,
        )
        return response.output_text
    except Exception as e:
        return f"呼叫 API 發生錯誤：{e}"


st.title("AI 問答 App")

user_input = st.text_input("請輸入問題：")
if st.button("送出") and user_input:
    with st.spinner("思考中..."):
        answer = ask_ai_safe(user_input)
    st.write(answer)


<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>🔑 為什麼要 <code>get_secret()</code>？</b><br>
本機開發時 key 放 <code>.env</code>；部署到雲端時 key 放 Streamlit Secrets。用同一個函式包起來，<b>程式碼完全不用改</b>就能兩邊都跑。<br>
第 8 節會詳細說明 <code>.env</code>、<code>.gitignore</code> 與雲端 Secrets 的設定。
</div>

執行方式一樣：`streamlit run app_2_ai.py`（記得先建立 `.env` 放入 `OPENAI_API_KEY`）。


## 5️⃣ 串流回覆：`st.write_stream()`

第 5 週我們用 `for event in stream:` 把回覆逐字印出來。在 Streamlit 裡，只要：

1. 寫一個 **generator**：用 `yield` 逐段吐出文字。
2. 把它交給 **`st.write_stream()`**，網頁就會像打字機一樣一個字一個字出現。

`st.write_stream()` 還會<b>回傳完整的文字</b>，方便我們接著存進對話紀錄。


In [ ]:
%%writefile app_3_stream.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI


def get_secret(name, default=None):
    try:
        if name in st.secrets:
            return st.secrets[name]
    except Exception:
        pass
    load_dotenv()
    return os.getenv(name, default)


def stream_ai(user_input, role="你是有幫助的 AI 助理，請用繁體中文回答。"):
    api_key = get_secret("OPENAI_API_KEY")
    if not api_key:
        yield "尚未設定 OPENAI_API_KEY。"
        return
    client = OpenAI(api_key=api_key)
    model = get_secret("OPENAI_MODEL", "gpt-5.4-mini")
    stream = client.responses.create(
        model=model,
        instructions=role,
        input=user_input,
        stream=True,
    )
    for event in stream:
        if getattr(event, "type", None) == "response.output_text.delta":
            yield event.delta


st.title("AI 問答 App（串流版）")

user_input = st.text_input("請輸入問題：")
if st.button("送出") and user_input:
    answer = st.write_stream(stream_ai(user_input))


<div style="background-color:#fff8e6; border-left:6px solid #f0ad4e; padding:12px 16px; border-radius:6px;">
<b>🔁 對照第 5 週</b>：<code>stream_ai()</code> 裡的 <code>for event in stream:</code> 判斷 <code>response.output_text.delta</code> 的寫法，和第 5 週<b>完全一樣</b>；差別只在第 5 週用 <code>print(..., end="")</code>，這週改成 <code>yield</code> 交給 <code>st.write_stream()</code> 顯示在網頁上。
</div>


## 6️⃣ 對話記憶：`st.session_state` + 聊天介面

還記得「整個腳本會重跑」嗎？所以聊天紀錄要存在 **`st.session_state`**（Streamlit 專門用來跨重跑保存資料的地方）。

搭配兩個聊天專用元件：
- **`st.chat_input()`**：畫面最下方的聊天輸入框。
- **`st.chat_message("user" / "assistant")`**：泡泡框，區分誰說的話。

流程：**每次重跑 → 先把歷史全部重畫一次 → 有新輸入才呼叫 AI 並存進歷史**。


In [ ]:
%%writefile app_4_chat.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI


def get_secret(name, default=None):
    try:
        if name in st.secrets:
            return st.secrets[name]
    except Exception:
        pass
    load_dotenv()
    return os.getenv(name, default)


def stream_ai(user_input, role="你是有幫助的 AI 助理，請用繁體中文回答。"):
    api_key = get_secret("OPENAI_API_KEY")
    if not api_key:
        yield "尚未設定 OPENAI_API_KEY。"
        return
    client = OpenAI(api_key=api_key)
    model = get_secret("OPENAI_MODEL", "gpt-5.4-mini")
    stream = client.responses.create(
        model=model, instructions=role, input=user_input, stream=True,
    )
    for event in stream:
        if getattr(event, "type", None) == "response.output_text.delta":
            yield event.delta


st.title("AI 聊天機器人")

# 1) 初始化對話紀錄（只在第一次執行時建立）
if "messages" not in st.session_state:
    st.session_state.messages = []

# 2) 每次重跑都先把過去對話重畫一次
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# 3) 讀取使用者的新輸入
prompt = st.chat_input("請輸入訊息")
if prompt:
    # 3a) 存入並顯示使用者訊息
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    # 3b) 串流顯示 AI 回覆，並存回歷史
    with st.chat_message("assistant"):
        answer = st.write_stream(stream_ai(prompt))
    st.session_state.messages.append({"role": "assistant", "content": answer})


<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>🧠 <code>st.session_state</code> 是什麼？</b><br>
它是一個像字典的容器，內容<b>在重跑之間不會消失</b>。除了聊天紀錄，之後也能用它存「累計花費」、「使用者設定」等需要記住的狀態。
</div>

> ⚠️ 這裡送給模型的是「這一輪的 `prompt`」。若要讓機器人記得**前幾輪內容**，可把整個 `st.session_state.messages` 組成上下文再送出（延伸自第 5 週的方法 B）。整合版 `app.py` 會示範。


## 7️⃣ 表單 `st.form`：一次送出多個欄位

如果一個功能有「多個輸入 + 一顆送出鈕」（例如：貼上文字 + 選風格 → 產生摘要），
直接用元件會有個困擾：**每改一個欄位，整個腳本就重跑一次**。

用 **`st.form`** 把欄位包起來，就會等到按下 **`st.form_submit_button`** 才一次送出，中途不重跑。


In [ ]:
# 這是可貼進 app 的片段示範（摘要小工具）
import streamlit as st

with st.form("summary_form"):
    source_text = st.text_area("貼上要摘要的文字", height=160)
    style = st.selectbox("摘要風格", ["條列重點", "一段式摘要", "給主管看的摘要"])
    submitted = st.form_submit_button("產生摘要")

if submitted and source_text.strip():
    prompt = f"請將以下內容整理成「{style}」：\n\n{source_text}"
    st.write(ask_ai_safe(prompt, role="你是嚴謹的中文摘要助理。"))


## 8️⃣ 檔案上傳：`st.file_uploader`

`st.file_uploader()` 讓使用者上傳檔案。這裡示範上傳 `.txt` / `.md` 後做摘要。
上傳的檔案要用 `.read()` 讀成 bytes，再 `.decode("utf-8")` 轉成文字。


In [ ]:
# 可貼進 app 的片段示範（檔案摘要）
import streamlit as st

uploaded_file = st.file_uploader("上傳 .txt 或 .md 檔案", type=["txt", "md"])
if uploaded_file is not None:
    text = uploaded_file.read().decode("utf-8", errors="ignore")
    st.text_area("檔案內容預覽", text[:2000], height=200)
    if st.button("摘要這份檔案"):
        prompt = "請閱讀以下文字並列出 5 個重點：\n\n" + text[:12000]
        st.write(ask_ai_safe(prompt, role="你是文件摘要助理。"))


## 9️⃣ API Key 管理與部署上線

<div style="background-color:#fdeeee; border-left:6px solid #d9534f; padding:12px 16px; border-radius:6px;">
<b>🔐 鐵則：API key 絕不寫進程式碼、絕不推上 GitHub。</b>
</div>

一個可部署的 Streamlit 專案，資料夾長這樣：

```
week07_streamlit_app/
├── app.py                ← 主程式
├── requirements.txt      ← 套件清單（雲端靠它安裝）
├── .gitignore            ← 告訴 Git 忽略 .env 等檔案
├── .env                  ← 本機用的 API key（不上傳！）
└── README.md             ← 專案說明
```

下面用 `%%writefile` 產生這三個設定檔（本機專案根目錄會用到）。


In [ ]:
%%writefile requirements.txt
streamlit
openai
python-dotenv


In [ ]:
%%writefile .gitignore
# API key 與環境變數（絕不上傳）
.env
*.env
.streamlit/secrets.toml

# Python 暫存
__pycache__/
*.pyc
.venv/

# 系統檔
.DS_Store
Thumbs.db


In [ ]:
%%writefile .env.example
# 複製本檔為 .env，填入自己的 key。.env 已被 .gitignore 忽略。
OPENAI_API_KEY=sk-proj-your-key-here
OPENAI_MODEL=gpt-5.4-mini


### 🚀 部署到 Streamlit Community Cloud（免費）

1. 把專案推上 **GitHub**（先確認 `.env` 沒有出現在變更清單中）。
2. 到 <https://share.streamlit.io> → 用 GitHub 登入 → **New app**。
3. 選 repo、branch 選 `main`、Main file path 填 `app.py` → **Deploy**。
4. 部署後到 App → **Settings → Secrets**，貼上：

```toml
OPENAI_API_KEY = "你的key"
OPENAI_MODEL = "gpt-5.4-mini"
```

5. 之後每次改程式，只要 **commit + push**，Streamlit 會自動重新部署。

> 📄 完整的 Git / GitHub Desktop 操作步驟，請搭配講義 **《第07週_Git實作教材.md》**。


## 🔟 整合版：完整的 `app.py`

以上功能（側邊欄設定、串流聊天、摘要表單、檔案上傳、累計成本）整合後的完整可部署版本，
放在教材附帶的專案資料夾：

```
week07_streamlit_app_claude/
├── app.py
├── requirements.txt
├── .gitignore
├── .env.example
├── README.md
└── .streamlit/config.toml
```

本機執行：

```bash
cd week07_streamlit_app_claude
pip install -r requirements.txt
# 複製 .env.example 為 .env 並填入 key
streamlit run app.py
```

> 👀 建議打開 `app.py` 對照本教材各節，理解每個功能是怎麼組起來的。


## 📝 課堂練習

以下三題請在<b>自己的 `app.py`</b> 中完成。教師版已附參考解答；學生版保留 TODO。

- 練習 A：在側邊欄加入可自訂的 system prompt（必做）
- 練習 B：做一個「翻譯 / 分類」小工具表單（必做）
- 練習 C：顯示本次對話累計 token 與估算成本（選做）


### 練習 A｜側邊欄自訂 system prompt

用 `st.sidebar` 加一個 `st.text_area` 讓使用者自訂 AI 的角色設定，
並把它當成 `role` 傳給 `ask_ai_safe()` 或 `stream_ai()`。


In [ ]:
# 練習 A 參考解答
import streamlit as st

with st.sidebar:
    st.header("設定")
    system_prompt = st.text_area(
        "System prompt（角色設定）",
        value="你是有幫助的 AI 助理，請用繁體中文回答。",
        height=120,
    )

user_input = st.text_input("請輸入問題：")
if st.button("送出") and user_input:
    answer = st.write_stream(stream_ai(user_input, role=system_prompt))


### 練習 B｜翻譯 / 分類小工具（表單）

用 `st.form` 做一個工具：使用者貼上文字，選擇要「翻成英文」或「判斷是正面/負面情緒」，
按送出後顯示結果。（提示：用 `st.selectbox` 選任務，組不同的 prompt。）


In [ ]:
# 練習 B 參考解答
import streamlit as st

with st.form("tool_form"):
    text = st.text_area("輸入文字", height=140)
    task = st.selectbox("選擇任務", ["翻譯成英文", "情緒分類（正面/負面）"])
    submitted = st.form_submit_button("執行")

if submitted and text.strip():
    if task == "翻譯成英文":
        prompt = f"請把以下文字翻譯成自然的英文，只輸出翻譯結果：\n\n{text}"
    else:
        prompt = f"請判斷以下文字的情緒是「正面」還是「負面」，只回答兩個字：\n\n{text}"
    st.write(ask_ai_safe(prompt))


### 練習 C（選做）｜顯示累計 token 與估算成本

延伸第 5 週練習 C。每次呼叫後從 `response.usage` 取得 `input_tokens` / `output_tokens`，
累加到 `st.session_state`，並在側邊欄顯示累計用量與估算成本。


In [ ]:
# 練習 C 參考解答（節錄重點）
import streamlit as st

PRICE_IN = 0.00015   # 美元 / 1K input tokens（示範用）
PRICE_OUT = 0.0006   # 美元 / 1K output tokens（示範用）

def accumulate(response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return
    st.session_state.total_in = st.session_state.get("total_in", 0) + usage.input_tokens
    st.session_state.total_out = st.session_state.get("total_out", 0) + usage.output_tokens

def show_cost():
    ti = st.session_state.get("total_in", 0)
    to = st.session_state.get("total_out", 0)
    cost = ti / 1000 * PRICE_IN + to / 1000 * PRICE_OUT
    st.sidebar.caption(f"累計：輸入 {ti} / 輸出 {to} tokens，約 US${cost:.5f}")

# 用法：在 ask_ai_safe() 內取得 response 後呼叫 accumulate(response)，
#      每次重跑時呼叫 show_cost()。


## ✅ 完成檢核清單

本週結束後，請確認你能夠：

- [ ] 用 `streamlit run app.py` 在本機啟動一個 Web App
- [ ] 說明「整個腳本每次互動都會重跑」以及為何要用 `st.session_state`
- [ ] 用 `st.write_stream()` 做出串流回覆
- [ ] 用 `st.chat_input` / `st.chat_message` 做出有記憶的聊天介面
- [ ] 用 `st.form` 與 `st.file_uploader` 做表單與檔案上傳
- [ ] 用 `.env`（本機）與 `st.secrets`（雲端）管理 API key，且 `.env` 沒有被 Git 追蹤
- [ ] （選做）把 App 部署到 Streamlit Community Cloud 並取得公開網址

### 📌 課後作業

1. 完成練習 A、B（C 選做），整合成自己的 `app.py`。
2. 建立 GitHub repo，設定好 `.gitignore`，完成至少兩次有意義的 commit。
3. （挑戰）部署到 Streamlit Community Cloud，把公開網址貼到課堂討論區。


---

<div style="background-color:#f5f5f5; border-left:6px solid #999; padding:12px 16px; border-radius:6px;">
🧩 <b>版本說明</b>：本教材由 <b>Claude Code</b> 產出（教師版）。<br>
主線：OpenAI Responses API + Streamlit；預設模型 <code>gpt-5.4-mini</code>（可用 <code>OPENAI_MODEL</code> 覆蓋）。<br>
下週預告：第 8 週起以 VS Code + GitHub 專案形式進行期中專題開發。
</div>
